# 中证800 V70 V46组合持仓数敏感性：top8 到 top20

这是独立版 notebook，可直接上传到 JoinQuant 研究环境运行。

它不依赖本地 `.py` 文件，也不依赖 `/Users/...` 路径。只需要把训练数据 CSV 放在当前目录或 `data/` 目录；如文件名不同，修改下面配置里的 `DATA_PATH_OVERRIDE`。

实验只比较组合层持仓数，不修改 V46 训练方式：`full/direct/fixed120/legacy_rebalance`。


## 0. 导入与进度条


In [ ]:
import os
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 260)
pd.set_option("display.width", 260)

try:
    from tqdm.auto import tqdm
except Exception:
    tqdm = None


def progress_iter(iterable, total=None, desc="progress", leave=True):
    if tqdm is not None:
        return tqdm(iterable, total=total, desc=desc, leave=leave)

    def _gen():
        every = max(1, int((total or 100) / 20))
        for i, item in enumerate(iterable, 1):
            if i == 1 or i % every == 0 or (total is not None and i == total):
                print("%s %s%s" % (desc, i, "/%s" % total if total else ""))
            yield item

    return _gen()


def display_df(df, n=30):
    try:
        display(df.head(n))
    except Exception:
        print(df.head(n).to_string(index=False))




## 1. 配置：路径、模型窗口、组合规格


In [ ]:
PROJECT_DIR = Path.cwd()
OUT_DIR = PROJECT_DIR / "csi800_ml_v70_portfolio_size_sensitivity_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_CANDIDATES = [
    Path("train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("train_csi800_factor_v40_data_enhancement.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement_20190101_20260531.csv"),
    Path("data/train_csi800_factor_v40_data_enhancement.csv"),
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "train_csi800_factor_v40_data_enhancement.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement_20190101_20260531.csv",
    PROJECT_DIR / "data" / "train_csi800_factor_v40_data_enhancement.csv",
]
DATA_PATH_OVERRIDE = None

TARGET_COL = "alpha_1m"
STOCK_COL = "stock"
DATE_COL = "rebalance_date"
INDUSTRY_COL = "industry_bucket"

FIXED_ITER = 120
SEED = 42
CORR_THRESHOLD = 0.70
LABEL_BOUNDARY_MODE = "legacy_rebalance"
RANDOM_SIM_N = 300
RANDOM_SEED = 42
SMOKE_TEST = False

DIAG_MODEL_SPECS = [
    {"model_tag": "exp_2021_12", "train_start": "2019-01-01", "train_end": "2021-12-31", "test_start": "2022-01-01", "test_end": "2023-12-31"},
    {"model_tag": "exp_2022_12", "train_start": "2019-01-01", "train_end": "2022-12-31", "test_start": "2023-01-01", "test_end": "2024-12-31"},
    {"model_tag": "exp_2023_12", "train_start": "2019-01-01", "train_end": "2023-12-31", "test_start": "2024-01-01", "test_end": "2025-12-31"},
    {"model_tag": "exp_2024_12", "train_start": "2019-01-01", "train_end": "2024-12-31", "test_start": "2025-01-01", "test_end": "2026-04-30"},
    {"model_tag": "exp_2025_12", "train_start": "2019-01-01", "train_end": "2025-12-31", "test_start": "2026-01-01", "test_end": "2026-06-30"},
]

if SMOKE_TEST:
    DIAG_MODEL_SPECS = DIAG_MODEL_SPECS[:1]

PORTFOLIO_SPECS = [
    {"portfolio_rule": "top8_cap3_2", "stock_num": 8, "board_caps": {"chinext": 3, "star": 2}, "family": "scaled_cap"},
    {"portfolio_rule": "top10_cap4_3", "stock_num": 10, "board_caps": {"chinext": 4, "star": 3}, "family": "scaled_cap"},
    {"portfolio_rule": "top12_cap5_3", "stock_num": 12, "board_caps": {"chinext": 5, "star": 3}, "family": "scaled_cap"},
    {"portfolio_rule": "top15_cap6_4", "stock_num": 15, "board_caps": {"chinext": 6, "star": 4}, "family": "scaled_cap"},
    {"portfolio_rule": "top20_cap8_5", "stock_num": 20, "board_caps": {"chinext": 8, "star": 5}, "family": "scaled_cap"},
    {"portfolio_rule": "top10_fixedcap3_2", "stock_num": 10, "board_caps": {"chinext": 3, "star": 2}, "family": "fixed_cap3_2"},
    {"portfolio_rule": "top12_fixedcap3_2", "stock_num": 12, "board_caps": {"chinext": 3, "star": 2}, "family": "fixed_cap3_2"},
    {"portfolio_rule": "top15_fixedcap3_2", "stock_num": 15, "board_caps": {"chinext": 3, "star": 2}, "family": "fixed_cap3_2"},
    {"portfolio_rule": "top20_fixedcap3_2", "stock_num": 20, "board_caps": {"chinext": 3, "star": 2}, "family": "fixed_cap3_2"},
]

MANUAL_FAILURE_MONTHS = [
    "2022-04-01", "2022-08-01", "2023-09-01", "2024-01-02", "2024-03-01",
    "2025-03-03", "2025-05-06", "2025-11-03", "2026-03-02",
]



## 2. V46 full 特征与 LightGBM 参数


In [ ]:
BASE_FACTOR_COLS = [
    "cash_flow_to_price_ratio", "book_to_price_ratio", "earnings_yield", "sales_to_price_ratio",
    "cash_earnings_to_price_ratio", "earnings_to_price_ratio", "roe_ttm", "roa_ttm",
    "gross_profit_ttm", "operating_profit_to_total_profit", "net_operate_cash_flow_to_total_liability",
    "net_operating_cash_flow_coverage", "adjusted_profit_to_total_profit", "ACCA", "growth",
    "net_working_capital", "operating_profit_per_share", "net_operate_cash_flow_per_share",
    "total_operating_revenue_per_share", "super_quick_ratio", "MLEV", "debt_to_equity_ratio",
    "debt_to_tangible_equity_ratio", "momentum", "Rank1M", "sharpe_ratio_60", "Variance20",
    "liquidity", "beta", "ATR6", "MFI14", "DAVOL10", "VOL10", "VMACD", "VOSC",
    "Skewness20", "Kurtosis20",
]

HYBRID_LIGHT_EXTRA_COLS = [
    "liq_money_ratio_20_60", "liq_paused_count_20", "px_close_to_ma60", "px_drawdown_60",
    "ts_cash_flow_to_price_ratio_rank_mean_3m", "ts_Rank1M_rank_chg_1m",
]

FULL_FEATURE_COLS = BASE_FACTOR_COLS + HYBRID_LIGHT_EXTRA_COLS

BASE_PARAMS_FF10 = {
    "objective": "regression",
    "metric": "l2",
    "boosting_type": "gbdt",
    "learning_rate": 0.05,
    "num_leaves": 31,
    "min_data_in_leaf": 200,
    "feature_fraction": 1.0,
    "bagging_fraction": 0.8,
    "bagging_freq": 1,
    "lambda_l1": 0.1,
    "lambda_l2": 0.3,
    "verbose": -1,
}




## 3. 工具函数：数据、训练、组合构建、统计


In [ ]:
def unique_keep_order(cols):
    seen = set()
    out = []
    for col in cols:
        if col not in seen:
            out.append(col)
            seen.add(col)
    return out


def resolve_data_path():
    if DATA_PATH_OVERRIDE:
        p = Path(DATA_PATH_OVERRIDE)
        if p.exists():
            return p
        raise IOError("DATA_PATH_OVERRIDE not found: %s" % p)
    candidates = [Path(x) for x in DATA_CANDIDATES]
    for p in candidates:
        if p.exists():
            return p
    searched = [str(p.resolve()) for p in candidates]
    raise IOError("training data csv not found. Put train_csi800_factor_v40_data_enhancement*.csv in one of: %s" % searched)


def safe_to_datetime(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce").dt.normalize()
    return out


def safe_rank_ic(a, b):
    s = pd.DataFrame({"a": np.asarray(a, dtype=float), "b": np.asarray(b, dtype=float)})
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) < 3 or s["a"].nunique() < 2 or s["b"].nunique() < 2:
        return np.nan
    return s["a"].rank(pct=True).corr(s["b"].rank(pct=True))


def calc_nav(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    if len(s) == 0:
        return pd.Series(dtype=float)
    return (1.0 + s).cumprod()


def calc_max_drawdown(ret_series):
    nav = calc_nav(ret_series)
    if len(nav) == 0:
        return np.nan
    return float((nav / nav.cummax() - 1.0).min())


def summarize_returns(ret_series):
    s = pd.Series(ret_series).replace([np.inf, -np.inf], np.nan).dropna()
    if len(s) == 0:
        return {"months": 0, "cum_ret": np.nan, "mean_ret": np.nan, "win_rate": np.nan, "max_drawdown": np.nan, "worst_month": np.nan}
    return {
        "months": int(len(s)),
        "cum_ret": float((1.0 + s).prod() - 1.0),
        "mean_ret": float(s.mean()),
        "win_rate": float((s > 0).mean()),
        "max_drawdown": calc_max_drawdown(s),
        "worst_month": float(s.min()),
    }


def build_corr_components(train_df, feature_cols, threshold):
    from collections import defaultdict
    corr = train_df[feature_cols].corr()
    graph = defaultdict(list)
    for i in range(len(feature_cols)):
        for j in range(i + 1, len(feature_cols)):
            v = corr.iloc[i, j]
            if not pd.isnull(v) and abs(v) > threshold:
                graph[feature_cols[i]].append(feature_cols[j])
                graph[feature_cols[j]].append(feature_cols[i])
    for col in feature_cols:
        graph[col]
    visited = set()
    comps = []

    def dfs(x, comp):
        visited.add(x)
        comp.append(x)
        for y in graph[x]:
            if y not in visited:
                dfs(y, comp)

    for col in feature_cols:
        if col not in visited:
            comp = []
            dfs(col, comp)
            comps.append(comp)
    return comps


def select_features_train_only(train_df, candidate_cols):
    cols = unique_keep_order([c for c in candidate_cols if c in train_df.columns])
    if len(cols) == 0:
        raise ValueError("no candidate feature exists in train data")
    missing = train_df[cols].isnull().sum().to_dict()
    keep = []
    remove = []
    for comp in build_corr_components(train_df, cols, CORR_THRESHOLD):
        if len(comp) == 1:
            keep.append(comp[0])
        else:
            comp = sorted(comp, key=lambda x: (missing[x], x))
            keep.append(comp[0])
            remove.extend(comp[1:])
    return keep, remove


def prepare_xy(df, feature_cols, target_col, fill_values=None):
    d = df.dropna(subset=[target_col]).copy()
    X = d.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    y = d[target_col].astype(float).copy()
    if fill_values is None:
        fill_values = X.median().replace([np.inf, -np.inf], np.nan).fillna(0)
    X = X.fillna(fill_values).fillna(0)
    return X, y, fill_values, d.index


def make_train_df(df_all, spec):
    start = pd.Timestamp(spec["train_start"])
    end = pd.Timestamp(spec["train_end"])
    mask = (df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)
    if LABEL_BOUNDARY_MODE == "label_end_safe":
        mask = mask & (df_all["next_date"] <= end)
    elif LABEL_BOUNDARY_MODE != "legacy_rebalance":
        raise ValueError("unknown LABEL_BOUNDARY_MODE: " + str(LABEL_BOUNDARY_MODE))
    return df_all[mask].copy()


def make_test_df(df_all, spec):
    start = pd.Timestamp(spec["test_start"])
    end = pd.Timestamp(spec["test_end"])
    return df_all[(df_all[DATE_COL] >= start) & (df_all[DATE_COL] <= end)].copy()


def train_direct_lgb(train_df, feature_cols):
    params = dict(BASE_PARAMS_FF10)
    params["seed"] = SEED
    X_train, y_train, fill_values, _ = prepare_xy(train_df, feature_cols, TARGET_COL)
    if len(X_train) == 0:
        raise ValueError("empty training matrix")
    model = lgb.train(params, lgb.Dataset(X_train, label=y_train), num_boost_round=max(1, int(FIXED_ITER)))
    pred = np.asarray(model.predict(X_train[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)
    return {"model": model, "fill_values": fill_values, "train_rows": int(len(X_train)), "train_rank_ic": safe_rank_ic(y_train, pred)}


def score_with_model(df, model, feature_cols, fill_values):
    X = df.reindex(columns=feature_cols).replace([np.inf, -np.inf], np.nan).copy()
    X = X.fillna(fill_values).fillna(0)
    return np.asarray(model.predict(X[feature_cols], num_iteration=FIXED_ITER)).reshape(-1)


def get_stock_board(stock):
    code = str(stock).split(".")[0]
    if code.startswith(("300", "301")):
        return "chinext"
    if code.startswith(("688", "689")):
        return "star"
    return "main"


def board_cap_allows(selected, stock, board_caps):
    board = get_stock_board(stock)
    if board not in board_caps:
        return True
    current = sum(1 for s in selected if get_stock_board(s) == board)
    return current < int(board_caps[board])


def build_board_capped_targets(sorted_stocks, target_num, board_caps):
    selected = []
    for stock in sorted_stocks:
        if stock in selected:
            continue
        if board_cap_allows(selected, stock, board_caps):
            selected.append(stock)
            if len(selected) >= int(target_num):
                return selected
    for stock in sorted_stocks:
        if stock not in selected:
            selected.append(stock)
            if len(selected) >= int(target_num):
                break
    return selected


def load_dataset(path):
    df = pd.read_csv(path)
    df = safe_to_datetime(df, [DATE_COL, "feature_date", "next_date"])
    if STOCK_COL not in df.columns:
        for alt in ["code", "security", "order_book_id"]:
            if alt in df.columns:
                df = df.rename(columns={alt: STOCK_COL})
                break
    if TARGET_COL not in df.columns:
        if "raw_return_1m" in df.columns and "benchmark_csi800_1m" in df.columns:
            df[TARGET_COL] = df["raw_return_1m"] - df["benchmark_csi800_1m"]
        else:
            raise ValueError("target column not found: " + TARGET_COL)
    if INDUSTRY_COL not in df.columns:
        df[INDUSTRY_COL] = "UNKNOWN"
    need = [STOCK_COL, DATE_COL, TARGET_COL, INDUSTRY_COL, "feature_date", "next_date"]
    missing = [c for c in need if c not in df.columns]
    if missing:
        raise ValueError("dataset missing columns: " + ",".join(missing))
    df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
    df = df.dropna(subset=[STOCK_COL, DATE_COL, TARGET_COL]).copy()
    return df


def random_percentile(month_df, selected, target_num, board_caps, n_sim=RANDOM_SIM_N, seed=RANDOM_SEED):
    if len(month_df) == 0 or len(selected) == 0:
        return np.nan
    rng = np.random.RandomState(seed)
    stocks = month_df[STOCK_COL].astype(str).tolist()
    ret_map = dict(zip(month_df[STOCK_COL].astype(str), pd.to_numeric(month_df[TARGET_COL], errors="coerce")))
    selected_ret = np.nanmean([ret_map.get(s, np.nan) for s in selected])
    if pd.isnull(selected_ret):
        return np.nan
    vals = []
    arr = np.arange(len(stocks))
    for _ in range(int(n_sim)):
        perm = rng.permutation(arr)
        ordered = [stocks[i] for i in perm]
        picked = build_board_capped_targets(ordered, target_num, board_caps)
        vals.append(np.nanmean([ret_map.get(s, np.nan) for s in picked]))
    if len(vals) == 0:
        return np.nan
    return float((np.asarray(vals) <= selected_ret).mean())


def summarize_group(df, keys):
    rows = []
    if len(df) == 0:
        return pd.DataFrame()
    grouped = df.groupby(keys)
    for name, gdf in grouped:
        if not isinstance(name, tuple):
            name = (name,)
        row = {}
        for i, key in enumerate(keys):
            row[key] = name[i]
        stats = summarize_returns(gdf["mean_alpha"])
        for k, v in stats.items():
            row[k] = v
        row["avg_monthly_alpha"] = float(pd.to_numeric(gdf["mean_alpha"], errors="coerce").mean())
        row["monthly_win_rate"] = float((pd.to_numeric(gdf["mean_alpha"], errors="coerce") > 0).mean())
        row["avg_random_alpha_percentile"] = float(pd.to_numeric(gdf["random_alpha_percentile"], errors="coerce").mean()) if "random_alpha_percentile" in gdf.columns else np.nan
        row["low_random_pct_rate"] = float((pd.to_numeric(gdf["random_alpha_percentile"], errors="coerce") < 0.25).mean()) if "random_alpha_percentile" in gdf.columns else np.nan
        row["avg_hit_true_top20"] = float(pd.to_numeric(gdf["hit_true_top20"], errors="coerce").mean()) if "hit_true_top20" in gdf.columns else np.nan
        row["avg_selected_bottom20_count"] = float(pd.to_numeric(gdf["selected_bottom20_count"], errors="coerce").mean()) if "selected_bottom20_count" in gdf.columns else np.nan
        row["avg_turnover"] = float(pd.to_numeric(gdf["turnover"], errors="coerce").mean()) if "turnover" in gdf.columns else np.nan
        rows.append(row)
    return pd.DataFrame(rows)




## 4. 运行实验并保存结果


In [ ]:
print("OUT_DIR:", OUT_DIR)
print("fixed V46 training:", LABEL_BOUNDARY_MODE, "full/direct/fixed", FIXED_ITER)
print("portfolio specs:", ",".join([x["portfolio_rule"] for x in PORTFOLIO_SPECS]))

DATA_PATH = resolve_data_path()
df_all = load_dataset(DATA_PATH)
print("DATA_PATH:", DATA_PATH)
print("loaded:", df_all.shape)
print("rebalance_date:", df_all[DATE_COL].min(), "->", df_all[DATE_COL].max())

score_panel_parts = []
model_meta_rows = []
for spec in progress_iter(DIAG_MODEL_SPECS, total=len(DIAG_MODEL_SPECS), desc="train V46 for V70"):
    train_df = make_train_df(df_all, spec)
    test_df = make_test_df(df_all, spec)
    if train_df.empty or test_df.empty:
        print("skip empty", spec["model_tag"], train_df.shape, test_df.shape)
        continue
    feature_cols, removed_cols = select_features_train_only(train_df, FULL_FEATURE_COLS)
    trained = train_direct_lgb(train_df, feature_cols)
    score_df = test_df.copy()
    score_df["score"] = score_with_model(score_df, trained["model"], feature_cols, trained["fill_values"])
    score_df["model_tag"] = spec["model_tag"]
    score_df["train_start"] = pd.Timestamp(spec["train_start"])
    score_df["train_end"] = pd.Timestamp(spec["train_end"])
    score_panel_parts.append(score_df)
    model_meta_rows.append({
        "model_tag": spec["model_tag"],
        "train_start": spec["train_start"],
        "train_end": spec["train_end"],
        "test_start": spec["test_start"],
        "test_end": spec["test_end"],
        "train_months": int(train_df[DATE_COL].nunique()),
        "train_rows": int(len(train_df)),
        "test_months": int(test_df[DATE_COL].nunique()),
        "test_rows": int(len(test_df)),
        "feature_count": int(len(feature_cols)),
        "removed_feature_count": int(len(removed_cols)),
        "train_rank_ic": trained["train_rank_ic"],
        "feature_cols": ",".join(feature_cols),
        "removed_features": ",".join(removed_cols),
    })
    print("trained", spec["model_tag"], "features", len(feature_cols), "train_ic", trained["train_rank_ic"])

score_panel_df = pd.concat(score_panel_parts, ignore_index=True) if score_panel_parts else pd.DataFrame()
model_meta_df = pd.DataFrame(model_meta_rows)
score_panel_df.to_csv(OUT_DIR / "v70_score_panel.csv", index=False)
model_meta_df.to_csv(OUT_DIR / "v70_model_meta.csv", index=False)

portfolio_rows = []
grouped_score = score_panel_df.groupby(["model_tag", DATE_COL]) if len(score_panel_df) else []
ngroups = score_panel_df.groupby(["model_tag", DATE_COL]).ngroups if len(score_panel_df) else 0
for (model_tag, dt), gdf in progress_iter(grouped_score, total=ngroups, desc="portfolio size sweep"):
    m = gdf.dropna(subset=["score", TARGET_COL]).copy()
    if m.empty:
        continue
    m[STOCK_COL] = m[STOCK_COL].astype(str)
    ordered = m.sort_values("score", ascending=False).reset_index(drop=True)
    ordered_stocks = ordered[STOCK_COL].tolist()
    ret_map = dict(zip(m[STOCK_COL], pd.to_numeric(m[TARGET_COL], errors="coerce")))
    true_top20 = set(m.sort_values(TARGET_COL, ascending=False).head(20)[STOCK_COL].astype(str).tolist())
    true_bottom20 = set(m.sort_values(TARGET_COL, ascending=True).head(20)[STOCK_COL].astype(str).tolist())
    rank_ic = safe_rank_ic(m["score"], m[TARGET_COL])
    for pspec in PORTFOLIO_SPECS:
        targets = build_board_capped_targets(ordered_stocks, pspec["stock_num"], pspec["board_caps"])
        target_rets = [ret_map.get(s, np.nan) for s in targets]
        board_counts = {"main": 0, "chinext": 0, "star": 0}
        for s in targets:
            b = get_stock_board(s)
            board_counts[b] = board_counts.get(b, 0) + 1
        row = {
            "model_tag": model_tag,
            "rebalance_date": dt,
            "portfolio_rule": pspec["portfolio_rule"],
            "family": pspec["family"],
            "stock_num": int(pspec["stock_num"]),
            "board_caps": ";".join(["%s:%s" % (k, pspec["board_caps"][k]) for k in sorted(pspec["board_caps"])]),
            "mean_alpha": float(np.nanmean(target_rets)) if len(target_rets) else np.nan,
            "median_alpha": float(np.nanmedian(target_rets)) if len(target_rets) else np.nan,
            "win_rate": float((pd.Series(target_rets) > 0).mean()) if len(target_rets) else np.nan,
            "rank_ic": rank_ic,
            "random_alpha_percentile": random_percentile(m, targets, pspec["stock_num"], pspec["board_caps"]),
            "hit_true_top20": len(set(targets) & true_top20) / float(max(1, len(targets))),
            "selected_bottom20_count": int(len(set(targets) & true_bottom20)),
            "board_main": int(board_counts.get("main", 0)),
            "board_chinext": int(board_counts.get("chinext", 0)),
            "board_star": int(board_counts.get("star", 0)),
            "targets": ",".join(targets),
        }
        portfolio_rows.append(row)

portfolio_monthly_df = pd.DataFrame(portfolio_rows)
if len(portfolio_monthly_df):
    portfolio_monthly_df = portfolio_monthly_df.sort_values(["model_tag", "portfolio_rule", "rebalance_date"]).reset_index(drop=True)
    turnover_rows = []
    for (model_tag, rule), gdf in portfolio_monthly_df.groupby(["model_tag", "portfolio_rule"]):
        prev = None
        for idx, row in gdf.sort_values("rebalance_date").iterrows():
            cur = set(str(row["targets"]).split(",")) if str(row["targets"]) else set()
            if prev is None or len(cur) == 0:
                turnover = np.nan
                overlap = np.nan
            else:
                overlap = len(cur & prev) / float(max(1, len(cur)))
                turnover = 1.0 - overlap
            turnover_rows.append({"idx": idx, "overlap_prev": overlap, "turnover": turnover})
            prev = cur
    turnover_df = pd.DataFrame(turnover_rows).set_index("idx") if turnover_rows else pd.DataFrame()
    if len(turnover_df):
        portfolio_monthly_df["overlap_prev"] = turnover_df["overlap_prev"]
        portfolio_monthly_df["turnover"] = turnover_df["turnover"]

portfolio_summary_df = summarize_group(portfolio_monthly_df, ["model_tag", "family", "portfolio_rule", "stock_num"])
robust_monthly_df = portfolio_monthly_df[portfolio_monthly_df["model_tag"].isin(["exp_2023_12", "exp_2024_12", "exp_2025_12"])].copy()
robust_summary_df = summarize_group(robust_monthly_df, ["family", "portfolio_rule", "stock_num"])

pair_rows = []
for model_tag, gdf in portfolio_summary_df.groupby("model_tag") if len(portfolio_summary_df) else []:
    base = gdf[gdf["portfolio_rule"] == "top8_cap3_2"]
    if len(base) == 0:
        continue
    base = base.iloc[0]
    for _, row in gdf.iterrows():
        out = row.to_dict()
        out["delta_cum_vs_top8"] = row["cum_ret"] - base["cum_ret"]
        out["delta_worst_vs_top8"] = row["worst_month"] - base["worst_month"]
        out["delta_mdd_vs_top8"] = row["max_drawdown"] - base["max_drawdown"]
        out["delta_win_rate_vs_top8"] = row["monthly_win_rate"] - base["monthly_win_rate"]
        out["delta_low_random_rate_vs_top8"] = row["low_random_pct_rate"] - base["low_random_pct_rate"]
        pair_rows.append(out)
pairwise_vs_top8_df = pd.DataFrame(pair_rows)

portfolio_monthly_df.to_csv(OUT_DIR / "v70_portfolio_size_monthly.csv", index=False)
portfolio_summary_df.to_csv(OUT_DIR / "v70_portfolio_size_summary.csv", index=False)
robust_summary_df.to_csv(OUT_DIR / "v70_robust_phase_portfolio_size_summary.csv", index=False)
pairwise_vs_top8_df.to_csv(OUT_DIR / "v70_pairwise_vs_top8.csv", index=False)

print("saved outputs:")
for fp in sorted(OUT_DIR.glob("v70_*.csv")):
    print("-", fp)

print("\nsummary:")
display_df(portfolio_summary_df.sort_values(["model_tag", "family", "stock_num"]), 60)
print("\nrobust phase summary:")
display_df(robust_summary_df.sort_values(["family", "stock_num"]), 40)
print("\npairwise vs top8:")
display_df(pairwise_vs_top8_df.sort_values(["model_tag", "family", "stock_num"]), 80)
